# Prediction Modeling of Non-Stationary EV Charging Demand Using ACN-Data
## Phase 1: Data Acquisition, Inspection & Exploratory Data Analysis (EDA)

**Project Context (B.Tech Time Series Analysis):**
Electric Vehicle (EV) charging behavior is inherently **non-stationary**. Demand fluctuates dynamically based on:
- Commuting schedules (arrival and departure hours)
- Day-of-week patterns (weekday campus occupancy vs. weekends/holidays)
- Seasonal shifts and academic calendar fluctuations
- Grid capacity and charging speed limits

In Phase 1, we inspect authentic session data from the **Caltech Adaptive Charging Network (ACN-Data)**, examine data completeness and schemas, engineer preliminary duration features, and observe non-stationary time-series patterns.

### 1. Import Required Libraries

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Define project paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Path: {DATA_RAW}")

### 2. Load the ACN-Data Charging Sessions

If you have not downloaded the dataset yet:
1. Obtain a free token at https://ev.caltech.edu/dataset
2. Run in terminal: `python scripts/download_acn_data.py --site caltech --start 2020-01-01 --end 2020-03-31`
3. Or place your raw `.json` / `.csv` file directly into `data/raw/`.

In [ ]:
# Check for processed session data first, then fallback to raw
processed_parquet = DATA_PROCESSED / 'caltech_sessions_summary.parquet'
processed_csv = DATA_PROCESSED / 'caltech_sessions_summary.csv'
csv_path = DATA_RAW / 'acn_sessions.csv'
json_path = DATA_RAW / 'acn_sessions.json'

if processed_parquet.exists():
    print(f'Loading processed session summary: {processed_parquet}')
    df_raw = pd.read_parquet(processed_parquet)
elif processed_csv.exists():
    print(f'Loading processed CSV: {processed_csv}')
    df_raw = pd.read_csv(processed_csv)
elif csv_path.exists():
    print(f'Loading data from CSV: {csv_path}')
    df_raw = pd.read_csv(csv_path)
elif json_path.exists():
    print(f'Loading data from JSON: {json_path}')
    with open(json_path, 'r', encoding='utf-8') as f:
        sessions = json.load(f)
    df_raw = pd.json_normalize(sessions)
else:
    raise FileNotFoundError(
        f'Dataset not found. Run python scripts/process_time_series.py to extract data.'
    )

print(f'Total sessions loaded: {len(df_raw):,}')
print(f'DataFrame dimensions: {df_raw.shape}')

### 3. Initial Dataset Inspection
Examine columns, non-null counts, data types, and preview records.

In [ ]:
# Display DataFrame schema
df_raw.info()

In [ ]:
# First 5 records
df_raw.head()

In [ ]:
# Missing value analysis
missing_summary = pd.DataFrame({
    'missing_count': df_raw.isnull().sum(),
    'missing_percentage': (df_raw.isnull().sum() / len(df_raw)) * 100
})
missing_summary[missing_summary['missing_count'] > 0].sort_values(by='missing_count', ascending=False)

### 4. Timestamp Preprocessing & Feature Engineering
Convert timestamps to UTC datetime objects and derive:
- **`connection_duration_hrs`**: Total hours vehicle was plugged in.
- **`charging_duration_hrs`**: Hours vehicle was actively drawing power.
- **`idle_duration_hrs`**: Hours vehicle remained parked after reaching full charge.
- **Calendar features**: `hour`, `day_of_week`, `is_weekend`.

In [ ]:
df = df_raw.copy()

# Convert timestamp strings to datetime
timestamp_cols = ['connectionTime', 'disconnectTime', 'doneChargingTime']
for col in timestamp_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

# Calculate durations in hours
if 'connectionTime' in df.columns and 'disconnectTime' in df.columns:
    df['connection_duration_hrs'] = (df['disconnectTime'] - df['connectionTime']).dt.total_seconds() / 3600.0

if 'connectionTime' in df.columns and 'doneChargingTime' in df.columns:
    df['charging_duration_hrs'] = (df['doneChargingTime'] - df['connectionTime']).dt.total_seconds() / 3600.0

if 'connection_duration_hrs' in df.columns and 'charging_duration_hrs' in df.columns:
    df['idle_duration_hrs'] = df['connection_duration_hrs'] - df['charging_duration_hrs']

# Extract time features from connectionTime (arrival time)
if 'connectionTime' in df.columns:
    df['arrival_hour'] = df['connectionTime'].dt.hour
    df['arrival_dayofweek'] = df['connectionTime'].dt.day_name()
    df['arrival_date'] = df['connectionTime'].dt.date
    df['is_weekend'] = df['connectionTime'].dt.dayofweek >= 5

df[['connectionTime', 'disconnectTime', 'kWhDelivered', 'connection_duration_hrs', 'charging_duration_hrs']].head()

### 5. Summary Statistics of Key Numerical Features

In [ ]:
numeric_cols = [c for c in ['kWhDelivered', 'connection_duration_hrs', 'charging_duration_hrs', 'idle_duration_hrs'] if c in df.columns]
df[numeric_cols].describe().round(2)

### 6. Exploratory Data Visualizations

#### 6.1 Distribution of Energy Delivered (kWh)

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(df['kWhDelivered'].dropna(), bins=40, kde=True, color='teal')
plt.title('Distribution of Delivered Energy per Session (kWh)', fontsize=13, fontweight='bold')
plt.xlabel('Energy Delivered (kWh)')
plt.ylabel('Session Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_energy_delivered_distribution.png', dpi=300)
plt.show()

#### 6.2 Connection Duration vs. Charging Duration
Observing the idle time (vehicles plugged in after charging is complete).

In [ ]:
plt.figure(figsize=(9, 4))
sns.kdeplot(df['connection_duration_hrs'].dropna(), label='Total Connection Duration (hrs)', color='#2b5c8f', fill=True, alpha=0.3)
sns.kdeplot(df['charging_duration_hrs'].dropna(), label='Active Charging Duration (hrs)', color='#e05638', fill=True, alpha=0.3)
plt.xlim(0, 24)
plt.title('Comparison: Total Connection Time vs Active Charging Time', fontsize=13, fontweight='bold')
plt.xlabel('Duration (Hours)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_durations_comparison.png', dpi=300)
plt.show()

#### 6.3 Arrival Time Profile by Hour of Day
Investigating rush hour plug-in times on the Caltech campus.

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='arrival_hour', hue='is_weekend', palette='Set2')
plt.title('EV Arrival Frequency by Hour of Day (Weekday vs Weekend)', fontsize=13, fontweight='bold')
plt.xlabel('Arrival Hour (24-hour UTC/Local)')
plt.ylabel('Session Count')
plt.legend(title='Is Weekend', labels=['Weekday', 'Weekend'])
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_arrivals_by_hour.png', dpi=300)
plt.show()

### 7. Time Series Aggregation & Non-Stationarity Inspection
To forecast charging demand, we aggregate the discrete charging sessions into a continuous daily time series of total energy demanded ($kWh_{daily}$).

In [ ]:
daily_demand = df.groupby('arrival_date')['kWhDelivered'].agg(['sum', 'count']).reset_index()
daily_demand.rename(columns={'sum': 'total_kWh', 'count': 'session_count'}, inplace=True)
daily_demand['arrival_date'] = pd.to_datetime(daily_demand['arrival_date'])
daily_demand.sort_values(by='arrival_date', inplace=True)

plt.figure(figsize=(12, 5))
plt.plot(daily_demand['arrival_date'], daily_demand['total_kWh'], marker='o', markersize=3, color='#1f77b4', linewidth=1.5)
plt.title('Daily EV Charging Energy Demand Time Series (Caltech)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Total Daily Energy (kWh)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_daily_energy_timeseries.png', dpi=300)
plt.show()

### 8. Observations on Non-Stationarity & Phase 1 Findings
1. **Seasonality & Periodicity**: Strong 7-day cyclical patterns are clearly visible (weekdays vs. weekends).
2. **Distribution Skewness**: Energy delivered per session is right-skewed with a long tail.
3. **Idle Capacity**: Vehicles stay connected significantly longer than their active charging period.
4. **Trend / Level Shifts**: Campus holidays, academic terms, and external shocks (e.g., Spring 2020 COVID disruptions) introduce dramatic shifts in mean and variance, which characterize **non-stationary** behavior.

**Next Steps for Phase 2 (Upcoming):**
- Formal stationarity tests (Augmented Dickey-Fuller & KPSS)
- Regular 1-hour interval time series resampling
- Benchmark baselines (ARIMA, Prophet) vs. Foundation Models (Chronos, TimesFM, or Lag-Llama).

### 7. Continuous Hourly Power Demand & Statistical Stationarity Tests
Now load the continuous 1-hour facility demand time series and formally test for non-stationarity using the **Augmented Dickey-Fuller (ADF)** test and the **Kwiatkowski-Phillips-Schmidt-Shin (KPSS)** test.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

hourly_path = DATA_PROCESSED / 'caltech_hourly_demand.parquet'
if hourly_path.exists():
    df_hourly = pd.read_parquet(hourly_path)
    print(f'Loaded hourly demand: {len(df_hourly):,} hours')
    
    # Run ADF Test
    adf_res = adfuller(df_hourly['avg_power_kw'].dropna())
    print(f'ADF Test Statistic: {adf_res[0]:.4f} (p-value: {adf_res[1]:.4e})')
    
    # Run KPSS Test
    kpss_res = kpss(df_hourly['avg_power_kw'].dropna(), regression='c', nlags='auto')
    print(f'KPSS Test Statistic: {kpss_res[0]:.4f} (p-value: {kpss_res[1]:.4f})')
    print('KPSS Critical values:', kpss_res[3])
    if kpss_res[1] < 0.05:
        print('-> Null hypothesis of stationarity REJECTED (p < 0.05). Non-stationarity statistically confirmed!')
else:
    print('Hourly demand parquet not found. Run scripts/process_time_series.py first.')